# CAP08-Joins

In [ ]:
from pyspark.shell import spark

person = spark.createDataFrame([
    (0, "Bill Chambers", 0, [100]),
    (1, "Matei Zaharia", 1, [500, 250, 100]),
    (2, "Michael Armbrust", 1, [250, 100])]) \
    .toDF("id", "name", "graduate_program", "spark_status")

graduateProgram = spark.createDataFrame([
    (0, "Masters", "School of Information", "UC Berkeley"),
    (2, "Masters", "EECS", "UC Berkeley"),
    (1, "Ph.D.", "EECS", "UC Berkeley")]) \
    .toDF("id", "degree", "department", "school")

sparkStatus = spark.createDataFrame([
    (500, "Vice President"),
    (250, "PMC Member"),
    (100, "Contributor")]) \
    .toDF("id", "status")

In [ ]:
person.createOrReplaceTempView("person")
graduateProgram.createOrReplaceTempView("graduateProgram")
sparkStatus.createOrReplaceTempView("sparkStatus")

### Example 1: Inner Joins

In [ ]:
joinExpression = person["graduate_program"] == graduateProgram["id"]

person.join(graduateProgram, person["graduate_program"] == graduateProgram["id"]).show()

In [ ]:
wrongJoinExpression = person["name"] == graduateProgram["school"]

person.join(graduateProgram, person["name"] == graduateProgram["school"]).show()

In [ ]:
spark.sql("""
    SELECT *
    FROM
        person AS per
          JOIN graduateProgram AS gpr
            ON (per.graduate_program = gpr.id)
""").show()

In [ ]:
joinType = "inner"

person.join(graduateProgram, joinExpression, joinType).show()

### Example 2: Outer Joins

In [ ]:
joinType = "outer"

person.join(graduateProgram, joinExpression, joinType).show()

In [ ]:
spark.sql("""
    SELECT *
    FROM
        person AS per
          FULL OUTER JOIN graduateProgram AS gpr
            ON (per.graduate_program = gpr.id)
""").show()

### Example 3: Left Outer Joins

In [ ]:
joinType = "left_outer"

person.join(graduateProgram, joinExpression, joinType).show()

In [ ]:
spark.sql("""
    SELECT *
    FROM
        person AS per
          LEFT JOIN graduateProgram AS gpr
            ON (per.graduate_program = gpr.id)
""").show()

### Example 4: Right Outer Joins

In [ ]:
joinType = "right_outer"

person.join(graduateProgram, joinExpression, joinType).show()

In [ ]:
spark.sql("""
    SELECT *
    FROM
        person AS per
          RIGHT JOIN graduateProgram AS gpr
            ON (per.graduate_program = gpr.id)
""").show()

### Example 5: Left Semi Joins

In [ ]:
joinType = "left_semi"

graduateProgram.join(person, joinExpression, joinType).show()

In [ ]:
gradProgram2 = graduateProgram.union(spark.createDataFrame([
    (0, "Masters", "Duplicate Row", "Duplicate School")]))

gradProgram2.createOrReplaceTempView("graduateProgram2")

gradProgram2.join(person, joinExpression, joinType).show()

### Example 6: Left Anti Joins

In [ ]:
joinType = "left_anti"

graduateProgram.join(person, joinExpression, joinType).show()

In [ ]:
spark.sql("""
    SELECT *
    FROM
        graduateProgram AS gpr
          LEFT ANTI JOIN person AS per
            ON (gpr.id = per.graduate_program)
""").show()

### Example 7: Natural Joins

In [ ]:
spark.sql("""
    SELECT *
    FROM
        graduateProgram NATURAL JOIN person
""").show()

### Example 8: Cross (Cartesian) Joins

In [ ]:
joinType = "cross"

graduateProgram.join(person, joinExpression, joinType).show()

In [ ]:
spark.sql("""
    SELECT *
    FROM
        graduateProgram AS gpr
          CROSS JOIN person AS per
            ON (gpr.id = per.graduate_program)
""").show()

In [ ]:
spark.sql("""
    SELECT *
    FROM graduateProgram CROSS JOIN person
""").show()

### Example 9: Joins on Complex Types

In [ ]:
from pyspark.sql.functions import expr

person.withColumnRenamed("id", "personId") \
    .join(sparkStatus, expr("array_contains(spark_status, id)")) \
    .show()

In [ ]:
spark.sql("""
    SELECT *
    FROM
        (SELECT id AS personId, name, graduate_program, spark_status FROM person)
            JOIN sparkStatus
                ON array_contains(spark_status, id)
""").show()

# End